# Lesson 3: Least squares. Fitting a line through noisy measurements (Legendre 1805, Gauss 1809)

## The old idea

On New Year's Day 1801 an astronomer spotted a new object (the dwarf planet Ceres) and tracked it for 41 days before it vanished behind the Sun. Everyone wanted to know where it would reappear. The observations were all slightly wrong (shaky hands, atmosphere), and every pair of observations suggested a slightly different orbit. **Which orbit do you trust?**

Adrien-Marie Legendre published the answer in 1805: choose the curve that makes the **sum of the squared misses** as small as possible. Carl Friedrich Gauss, who used Ceres to demonstrate it (and said he'd had the method since 1795), published his version in 1809. It was the first systematic way to turn many inconsistent measurements into one best answer.

## What it becomes in machine learning

This *is* linear regression, the first algorithm in nearly every ML course. There is no "AI" in it: a 220-year-old calculus problem. The same "minimise squared misses" recipe underlies how neural networks are trained today.

**By the end of this lesson you can:** define the sum of squared errors, derive the best-fit line **exactly** using the slope tools from Lessons 1-2 (set the slopes to zero and solve), name the pieces (mean, variance, covariance), and verify the answer three independent ways.

## Our measurements: Hooke's law (1660s)

Robert Hooke found that a spring stretches in proportion to the weight hung on it. Here are five (invented, but realistic) measurements: a spring stretched by hanging 1, 2, 3, 4, 5 kg:

*Setup cell: the tools from Lessons 1 and 2. Run as-is.*

In [ ]:
def slope_by_wiggling(function, point, gap=0.0001):
    return (function(point + gap) - function(point)) / gap

def partial_derivative(function, point, which_input, gap=0.0001):
    wiggled_point = list(point)
    wiggled_point[which_input] = wiggled_point[which_input] + gap
    return (function(wiggled_point) - function(point)) / gap

def gradient(function, point):
    return [partial_derivative(function, point, index) for index in range(len(point))]

```python
masses_in_kg = [1, 2, 3, 4, 5]
stretches_in_cm = [2.1, 3.9, 6.2, 7.8, 10.1]
```

In [ ]:
# type it here

Hooke's law says `stretch = k × mass` for some constant `k`, a straight line through zero. The measurements are noisy, so no single `k` fits them all exactly. Which `k` is best?

---
## Step 1: How wrong is a guess? The sum of squared errors

Pick a guess for `k`. It predicts `k × mass` for each mass. The **error** (or *residual*) at each point is prediction minus measurement. Why *square* the errors before adding them up?

- Plain errors cancel: a +1 and a −1 sum to 0, which would look perfect.
- Squaring makes every error positive, **and punishes big misses much more** than small ones (a miss of 3 costs 9, three misses of 1 cost 3).
- Squaring gives a smooth curve with a nice slope everywhere (Lesson 2 showed that `abs` has a kink). That matters in a moment.

Type:

```python
def sum_of_squared_errors(spring_constant, masses, stretches):
    total = 0
    for mass, measured_stretch in zip(masses, stretches):
        predicted_stretch = spring_constant * mass
        error = predicted_stretch - measured_stretch
        total = total + error ** 2
    return total

for guess in [1.5, 2.0, 2.5]:
    print(guess, sum_of_squared_errors(guess, masses_in_kg, stretches_in_cm))
```

In [ ]:
# type it here

**Expected output:**
```text
1.5 14.059999999999999
2.0 0.1100000000000001
2.5 13.660000000000002
```

In a textbook that function is written:

```
S(k) = Σ (k·xᵢ − yᵢ)²
```

`Σ` (capital sigma, "sum") means "add up the following for every data point `i`". `xᵢ` is the i-th mass, `yᵢ` the i-th measured stretch. `S` is what your loop computes. `k = 2.0` scores best of the three here.

**Try this (new cell):** loop over `k` from 1.8 to 2.2 in steps of 0.05 (hint: `for step in range(...)` and `k = 1.8 + step * 0.05`). Where is the lowest score? What does the shape of the numbers look like as you move away from it?

---
## Step 2: S(k) is a parabola, so we can solve for its bottom

That "bowl" shape isn't an accident. Expand the square for one point: `(k·x − y)² = k²x² − 2kxy + y²`. Add that up over all points. Since `k` is the same for every point, it factors out of each sum:

```
S(k) = k² · Σx²  −  2k · Σxy  +  Σy²
```

That's `a·k² + b·k + c`: a **parabola** in `k`, with `a = Σx²` (positive, so it opens upward like a bowl). The three sums are just numbers computed once from the data. Check this identity in code:

```python
sum_of_x_squared = sum(mass ** 2 for mass in masses_in_kg)
sum_of_x_times_y = sum(mass * stretch for mass, stretch in zip(masses_in_kg, stretches_in_cm))
sum_of_y_squared = sum(stretch ** 2 for stretch in stretches_in_cm)

print(sum_of_x_squared, sum_of_x_times_y, sum_of_y_squared)

guess = 2.5
expanded_version = guess ** 2 * sum_of_x_squared - 2 * guess * sum_of_x_times_y + sum_of_y_squared
print(expanded_version)
print(sum_of_squared_errors(guess, masses_in_kg, stretches_in_cm))
```

In [ ]:
# type it here

**Expected output:**
```text
55 110.2 220.91
13.659999999999997
13.660000000000002
```

Both versions agree (13.66 for `k=2.5`, the same score you saw in Step 1). Now use Lesson 1: the bottom of a bowl is where the slope is zero. Slope of `S(k)` (power rule, Lesson 1):

```
dS/dk = 2k · Σx²  −  2 · Σxy  =  0
⟹  k = Σxy / Σx²
```

**That's the answer, exactly, in one line.** No searching. This is the closed-form **least squares** solution for a line through the origin. Compute it and check it two ways: against a brute-force search, and by confirming the slope of `S` there really is ~0 using your Lesson 1 tool.

```python
best_spring_constant = sum_of_x_times_y / sum_of_x_squared
print(best_spring_constant)

# check 1: brute force over a fine grid
best_so_far = None
for step in range(0, 4001):
    candidate = step * 0.001
    score = sum_of_squared_errors(candidate, masses_in_kg, stretches_in_cm)
    if best_so_far is None or score < best_so_far[1]:
        best_so_far = (candidate, score)
print(best_so_far)

# check 2: the slope of S at the answer should be about 0
score_function = lambda k: sum_of_squared_errors(k, masses_in_kg, stretches_in_cm)
print(slope_by_wiggling(score_function, best_spring_constant))
```

In [ ]:
# type it here

**Expected output:**
```text
2.003636363636364
(2.004, 0.10928000000000004)
0.005499999999047933
```

The formula gives `k ≈ 2.0036`; brute force lands on the same value (to 3 decimals, 2.004); and the slope of the error at that `k` is about 0 (the tiny leftover is the wiggle's small bias). Three independent routes agree.

---
## Step 3: Add an intercept (two knobs)

Real data usually doesn't pass through the origin. Allow a line `prediction = w·x + b` with **two** knobs: slope `w` and intercept `b`. Now the error is a function of two inputs, `S(w, b)`: a bowl-shaped *surface*. Its bottom is where **both partial slopes** (Lesson 2) are zero. Chain rule, from Lesson 2's "2 × error × input" sentence:

```
∂S/∂b = Σ 2·(w·xᵢ + b − yᵢ) · 1          = 0
∂S/∂w = Σ 2·(w·xᵢ + b − yᵢ) · xᵢ         = 0
```

Solve them. From the first (divide by 2 and expand): `w·Σx + n·b − Σy = 0`. Divide by `n` (the number of points) and write `x̄ = Σx/n`, `ȳ = Σy/n` for the **means**:

```
b = ȳ − w·x̄          (★)
```

In words: **the best line always passes through the point (mean x, mean y).** Now substitute (★) into the second equation. After tidying (try it on paper; the terms rearrange into sums of deviations from the means) you get:

```
w = Σ (xᵢ − x̄)(yᵢ − ȳ)  /  Σ (xᵢ − x̄)²
```

The bottom is the sum of squared deviations of `x` from its mean: related to **variance** (variance = that sum divided by n). The top, the sum of *products* of deviations, is related to **covariance**: positive when x and y tend to rise together, negative when one rises as the other falls. So:

```
slope = covariance(x, y) / variance(x)
```

Type the pieces, one at a time:

```python
def mean(numbers):
    return sum(numbers) / len(numbers)

def sum_of_deviation_products(first_numbers, second_numbers):
    first_mean = mean(first_numbers)
    second_mean = mean(second_numbers)
    total = 0
    for first_value, second_value in zip(first_numbers, second_numbers):
        total = total + (first_value - first_mean) * (second_value - second_mean)
    return total

print(mean(masses_in_kg), mean(stretches_in_cm))
print(sum_of_deviation_products(masses_in_kg, stretches_in_cm))
print(sum_of_deviation_products(masses_in_kg, masses_in_kg))
```

In [ ]:
# type it here

**Expected output:**
```text
3.0 6.0200000000000005
19.9
10.0
```

`sum_of_deviation_products(x, x)` is the bottom of the slope formula: how spread out `x` is. Using the same function on `(x, y)` gives the top. Now assemble the best-fit line:

```python
def fit_line_by_least_squares(x_values, y_values):
    slope = sum_of_deviation_products(x_values, y_values) / sum_of_deviation_products(x_values, x_values)
    intercept = mean(y_values) - slope * mean(x_values)
    return slope, intercept

fitted_slope, fitted_intercept = fit_line_by_least_squares(masses_in_kg, stretches_in_cm)
print(fitted_slope, fitted_intercept)
```

In [ ]:
# type it here

**Expected output:**
```text
1.9899999999999998 0.0500000000000016
```

Slope `1.99`, intercept `0.05`. Verify the way the maths demands: **both partial slopes of S should be ≈ 0** at the answer (Lesson 2's `gradient`):

```python
def two_knob_error(knobs):
    slope, intercept = knobs
    total = 0
    for x_value, y_value in zip(masses_in_kg, stretches_in_cm):
        error = slope * x_value + intercept - y_value
        total = total + error ** 2
    return total

print(two_knob_error([fitted_slope, fitted_intercept]))
print(gradient(two_knob_error, [fitted_slope, fitted_intercept]))
print(two_knob_error([2.1, 0.12]), two_knob_error([1.9, 0.12]), two_knob_error([2.0, 0.3]))
```

In [ ]:
# type it here

**Expected output:**
```text
0.10700000000000072
[0.0054999999964111534, 0.0004999999959898194]
1.0279999999999998 0.388000000000001 0.5000000000000011
```

Both partial slopes are ~0 (tiny numbers from the wiggle's small bias), and nudging either knob away from the answer makes the error larger (1.03, 0.39, and 0.50, against 0.107 at the answer). The minimum error, `0.107`, is only slightly below the `0.109` of the through-the-origin line. That makes sense: Hooke's law really does pass through zero, so the extra knob barely helps here. With data that doesn't pass through zero it would matter a lot (see the challenge).

---
## Step 4: Wrap it in a class (the growing file starts here)

From now on we'll organise each model as an object that holds its knobs and knows how to predict and score. Type:

```python
class LeastSquaresLine:
    def __init__(self):
        self.slope = 0.0
        self.intercept = 0.0

    def fit(self, x_values, y_values):
        self.slope = sum_of_deviation_products(x_values, y_values) / sum_of_deviation_products(x_values, x_values)
        self.intercept = mean(y_values) - self.slope * mean(x_values)

    def predict(self, x_value):
        return self.slope * x_value + self.intercept

    def sum_of_squared_errors(self, x_values, y_values):
        total = 0
        for x_value, y_value in zip(x_values, y_values):
            total = total + (self.predict(x_value) - y_value) ** 2
        return total
```

In [ ]:
# type the class here

```python
spring_model = LeastSquaresLine()
spring_model.fit(masses_in_kg, stretches_in_cm)
print(spring_model.slope, spring_model.intercept)
print(spring_model.predict(6))
print(spring_model.sum_of_squared_errors(masses_in_kg, stretches_in_cm))
```

In [ ]:
# type it here

**Expected output:**
```text
1.9899999999999998 0.0500000000000016
11.989999999999998
0.10700000000000072
```

**How the object stores data:** `self.slope` and `self.intercept` live inside the object (start at 0.0, set by `fit`). `predict` uses them. Predicting a 6 kg load gives about 12.0 cm.

---
## Step 5: Guided exploration: what a single bad measurement does

Least squares has a weakness, and seeing it matters. Copy this into a *new* cell: add a wild measurement (someone bumped the spring: 6 kg stretched it only 4 cm) and refit.

```python
masses_with_bad_point = masses_in_kg + [6]
stretches_with_bad_point = stretches_in_cm + [4.0]

model_with_bad_point = LeastSquaresLine()
model_with_bad_point.fit(masses_with_bad_point, stretches_with_bad_point)
print(model_with_bad_point.slope, model_with_bad_point.intercept)
```

In [ ]:
# type it here

**Expected output:**
```text
0.8485714285714285 2.713333333333334
```

The slope dropped from `1.99` to about `0.85`, and the intercept jumped from `0.05` to `2.7`: one point out of six more than halved the slope. Squaring means one big miss dominates the total, so the line bends toward it. Whether that's what you want depends on whether the odd point is a real signal or a blunder.

**A named alternative:** Henri Theil (1950) and Pranab Sen (1968) proposed taking the slope between *every pair* of points and using the **median** of those slopes. A median ignores wild values. Type it and compare:

```python
def theil_sen_slope(x_values, y_values):
    pairwise_slopes = []
    for first_index in range(len(x_values)):
        for second_index in range(first_index + 1, len(x_values)):
            run = x_values[second_index] - x_values[first_index]
            rise = y_values[second_index] - y_values[first_index]
            pairwise_slopes.append(rise / run)
    pairwise_slopes.sort()
    middle = len(pairwise_slopes) // 2
    if len(pairwise_slopes) % 2 == 1:
        return pairwise_slopes[middle]
    return (pairwise_slopes[middle - 1] + pairwise_slopes[middle]) / 2

print(theil_sen_slope(masses_in_kg, stretches_in_cm))
print(theil_sen_slope(masses_with_bad_point, stretches_with_bad_point))
```

In [ ]:
# type it here

**Expected output:**
```text
1.975
1.8999999999999997
```

Theil-Sen barely moves (about 1.98 → 1.90) while least squares lurched. **Why does anyone still use least squares?** It has a one-line exact formula, a smooth bowl to walk down (Lesson 4), and, as Lesson 9 shows, it's the *provably best* choice when the noise is bell-curve shaped. Knowing the alternatives tells you when to be suspicious. Return to the original data for the next step.

---
## Step 6: Challenge

You run an ad campaign. Spend (in thousands of dollars) and resulting sales (thousands of units):

```
ad_spend = [1, 2, 3, 4]
sales    = [3, 5, 6, 10]
```

1. **On paper**: compute `x̄`, `ȳ`, the top sum `Σ(x−x̄)(y−ȳ)` and bottom sum `Σ(x−x̄)²`, then the slope and intercept.
2. Predict: what will `sum_of_squared_errors` be for your line? (Compute each residual by hand.)
3. In a new cell, use `LeastSquaresLine` to check slope, intercept, and total error, and use `gradient` on a two-knob error function to confirm both partial slopes ≈ 0.
4. A friend says "the line should also pass through the point (0, 0), no intercept". Compute the through-origin slope with `Σxy / Σx²` and compare total errors. Which is better and why must it be?

```python
# your attempt here
```

In [ ]:
# type it here

---
## Where this goes next

The exact solution here relied on algebra we could do by hand with two knobs. With 50 knobs the hand-algebra is hopeless, and with a neural network no closed form exists at all. Lesson 4 shows the other route, which always works: **walk downhill step by step**, which is Cauchy's 1847 method and the engine of all modern training. Lesson 5 shows how linear algebra turns the many-knob version of *this* lesson's formula into one tidy matrix equation.

Once you're comfortable: you derived `w = covariance / variance`. Search for "correlation coefficient" and see how it relates (it's the same covariance, rescaled so it always lands between −1 and 1). Then see whether you can use the correlation and the two spreads to rebuild the slope.

---
# Solutions to the Step 6 challenge (try it first)

**1.** `x̄ = 2.5`, `ȳ = 6`. Deviations of x: `−1.5, −0.5, 0.5, 1.5`; of y: `−3, −1, 0, 4`. Products: `4.5, 0.5, 0, 6` → top sum `11`. Squared x deviations: `2.25, 0.25, 0.25, 2.25` → bottom sum `5`. Slope `= 11/5 = 2.2`. Intercept `= 6 − 2.2·2.5 = 0.5`.

**2.** Predictions `2.7, 4.9, 7.1, 9.3`; residuals vs `3, 5, 6, 10`: `−0.3, −0.1, 1.1, −0.7`; squares `0.09, 0.01, 1.21, 0.49` → total `1.8`.

```python
ad_spend = [1, 2, 3, 4]
sales = [3, 5, 6, 10]

ad_model = LeastSquaresLine()
ad_model.fit(ad_spend, sales)
print(ad_model.slope, ad_model.intercept)
print(ad_model.sum_of_squared_errors(ad_spend, sales))

def ad_error(knobs):
    slope, intercept = knobs
    return sum((slope * x + intercept - y) ** 2 for x, y in zip(ad_spend, sales))

print(gradient(ad_error, [ad_model.slope, ad_model.intercept]))

through_origin_slope = sum(x * y for x, y in zip(ad_spend, sales)) / sum(x ** 2 for x in ad_spend)
print(through_origin_slope, ad_error([through_origin_slope, 0]))
```

**Output:**
```text
2.2 0.5
1.7999999999999998
[0.0029999999906493713, 0.0004000000020099037]
2.3666666666666667 1.966666666666666
```

**4.** The through-origin slope is `Σxy/Σx² = 71/30 ≈ 2.367` with total error about `1.97`, worse than the `1.8` of the free line. It *must* be: the through-origin line is one of the lines the two-knob search could have chosen (the one with intercept forced to 0), and the best line over all choices can never be worse than the best over a restricted set.